In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, f1_score, classification_report, roc_auc_score, average_precision_score
import numpy as np
PROJECT_ROOT = os.path.abspath("..")    
sys.path.append(PROJECT_ROOT)

In [ ]:

from dataset.otto_final import TraceOttoDataset
from utils.SplitData import split_data_Train_Val_Test_LSTM
from utils.feature_engineering import get_between_features, get_elapsed_feature
from utils.plot_confussion_matrix import plot_confusion_matrix
from utils.training_utils import append_probs_and_true, concate_probs_true, search_best_f1_thr
from model.BaseModel_BI_LSTM import Bi_LSTM
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
dataset_processed  = TraceOttoDataset(
    file_name='../train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=32,
)

train_loader, validation_loader, test_loader = split_data_Train_Val_Test_LSTM(dataset_processed, batch_size=128)


print(len(train_loader.dataset))
print(len(validation_loader.dataset))
print(len(test_loader.dataset))


In [ ]:
max_aid = max(
        session[0]["aid"].max().item()
        for session in dataset_processed
    )
max_type = max(
        session[0]["type"].max().item()
        for session in dataset_processed
)

num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
    
RNN_model = Bi_LSTM(num_embeddings_aid, num_embeddings_event_type, embedding_dim=32, hidden_size=128, num_layers=2).to(device)

#Need to be changed after the training, to have the model that was trained on
final_model_load = torch.load("Model_TRACE_MLT_ATC_SAT_MLP_FinalVersion_70_80_.pt", weights_only=False, map_location=device)

RNN_model.load_state_dict(final_model_load["model_state_dict"])
best_thr_ATC = final_model_load["best_global_threshold"]["ATC"]
best_thr_SAT = final_model_load["best_global_threshold"]["SAT"]
best_thr_MAP = final_model_load["best_global_threshold"]["MAP"]


print(
    f"Loaded model | ATC: {best_thr_ATC:.4f} | "
    f"SAT: {best_thr_SAT:.4f} | "
    f"MAP: {best_thr_MAP:.4f}"
)


In [ ]:

def build_time_feats(timestamps, L):
    """
    Ensure that time features have dimension (B, L, 1) so that they can be concatenated with embeddings and fed into BiLSTM without dimension errors.
    """
    B = timestamps.shape[0]
    de = get_elapsed_feature(timestamps).float().to(timestamps.device)
    db = get_between_features(timestamps).float().to(timestamps.device)

    if de.dim() == 1:
        de = de.view(B, 1, 1).expand(B, L, 1)
    elif de.dim() == 2:
        de = de.unsqueeze(-1) if de.shape[1] != 1 else de.view(B, 1, 1).expand(B, L, 1)

    if db.dim() == 2:
        db = db.unsqueeze(-1)
    if db.shape[1] == L - 1:
        db = torch.cat([db, db[:, -1:, :]], dim=1)

    return de, db

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()

test_loss = 0.0
y_true_ATC, y_prob_ATC, y_pred_ATC = [], [], []
y_true_SAT, y_prob_SAT, y_pred_SAT = [] , [], []
y_true_MAP, y_prob_MAP , y_pred_MAP = [] , [], []

RNN_model.eval()

with torch.no_grad():
    for inputs_test, lengths_test, targets_test in test_loader:
        inputs_test = {k: v.to(device) for k, v in inputs_test.items()}
        lengths_test = lengths_test.to(device)

        target_test_ATC = targets_test["ATC"].to(device).float()
        target_test_SAT = targets_test["SAT"].to(device).float()
        target_test_MAP = targets_test["MAP"].to(device).float()
        B, L = inputs_test["aid"].shape
        delta_elapsed, delta_between = build_time_feats(inputs_test["timestamps"], L)
                
        logits_test = RNN_model(
                    inputs_test["aid"].long(),
                    inputs_test["type"].long(),
                    delta_elapsed,
                    delta_between,
                    lengths_test
                )

        logits_ATC_test = logits_test["ATC"]  
        logits_SAT_test = logits_test["SAT"]  
        logits_MAP_test = logits_test["MAP"]
                    
                    
        loss_ATC_test = criterion(logits_ATC_test, target_test_ATC)
        loss_SAT_test = criterion(logits_SAT_test, target_test_SAT)
        loss_MAP_test = criterion(logits_MAP_test, target_test_MAP)
        

        loss_test = (loss_ATC_test + loss_SAT_test + loss_MAP_test) / 3.0
        test_loss += loss_test.item()

        append_probs_and_true(logits_ATC_test, target_test_ATC, y_prob_ATC, y_true_ATC)
        append_probs_and_true(logits_SAT_test, target_test_SAT, y_prob_SAT, y_true_SAT)
        append_probs_and_true(logits_MAP_test, target_test_MAP, y_prob_MAP, y_true_MAP)

test_loss /= len(test_loader)


probs_ATC, true_ATC = concate_probs_true(y_prob_ATC, y_true_ATC)
probs_SAT, true_SAT = concate_probs_true(y_prob_SAT, y_true_SAT)
probs_MAP, true_MAP = concate_probs_true(y_prob_MAP, y_true_MAP)


pred_ATC = (probs_ATC >= best_thr_ATC).astype(int)
pred_SAT = (probs_SAT >= best_thr_SAT).astype(int)
pred_MAP = (probs_MAP >= best_thr_MAP).astype(int)


f1_ATC = f1_score(true_ATC, pred_ATC, zero_division=0)
f1_SAT = f1_score(true_SAT, pred_SAT, zero_division=0)
f1_MAP = f1_score(true_MAP, pred_MAP, zero_division=0)

macro_f1_ATC = f1_score(true_ATC, pred_ATC, average="macro", zero_division=0)
macro_f1_SAT = f1_score(true_SAT, pred_SAT, average="macro", zero_division=0)
macro_f1_MAP = f1_score(true_MAP, pred_MAP, average="macro", zero_division=0)

auroc_ATC = roc_auc_score(true_ATC, probs_ATC) 
auroc_SAT = roc_auc_score(true_SAT, probs_SAT) 
auroc_MAP = roc_auc_score(true_MAP, probs_MAP) 

auprc_ATC = average_precision_score(true_ATC, probs_ATC)
auprc_SAT = average_precision_score(true_SAT, probs_SAT)
auprc_MAP = average_precision_score(true_MAP, probs_MAP)

cm_ATC = confusion_matrix(true_ATC, pred_ATC)
cm_SAT = confusion_matrix(true_SAT, pred_SAT)
cm_MAP = confusion_matrix(true_MAP, pred_MAP)



test_accuracy_ATC = (pred_ATC == true_ATC).mean()
test_accuracy_SAT = (pred_SAT == true_SAT).mean()
test_accuracy_MAP = (pred_MAP == true_MAP).mean()
test_accuracy_global = np.mean([test_accuracy_ATC, test_accuracy_SAT, test_accuracy_MAP])


f1_mean = (f1_ATC + f1_SAT + f1_MAP) / 3
macro_f1_mean = (macro_f1_ATC + macro_f1_SAT + macro_f1_MAP) / 3
auroc_mean = (auroc_ATC + auroc_SAT + auroc_MAP) / 3
auprc_mean = (auprc_ATC + auprc_SAT + auprc_MAP) / 3

print(f"Test accuracy global: {test_accuracy_global:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy mean: {np.mean([test_accuracy_ATC, test_accuracy_SAT, test_accuracy_MAP]):.4f}\n")

print(f"F1 Mean Score: {f1_mean:.4f}")
print(f"F1 ATC Score: {f1_ATC:.4f}")
print(f"F1 SAT Score: {f1_SAT:.4f}")
print(f"F1 MAP Score: {f1_MAP:.4f}\n")

print(f"Macro F1 ATC Score: {macro_f1_ATC:.4f}")
print(f"Macro F1 SAT Score: {macro_f1_SAT:.4f}")
print(f"Macro F1 MAP Score: {macro_f1_MAP:.4f}")
print(f"Macro F1 MEAN Score: {macro_f1_mean:.4f}\n")

print(f"AUROC ATC Score: {auroc_ATC:.4f}")
print(f"AUROC SAT Score: {auroc_SAT:.4f}")
print(f"AUROC MAP Score: {auroc_MAP:.4f}")
print(f"AUROC MEAN Score: {auroc_mean:.4f}\n")

print(f"AUPRC ATC Score: {auprc_ATC:.4f}")
print(f"AUPRC SAT Score: {auprc_SAT:.4f}")
print(f"AUPRC MAP Score: {auprc_MAP:.4f}")
print(f"AUPRC MEAN Score: {auprc_mean:.4f}\n")

print(f"Optimum Threshold ATC: {best_thr_ATC:.4f}")
print(f"Optimum Threshold SAT: {best_thr_SAT:.4f}")
print(f"Optimum Threshold MAP: {best_thr_MAP:.4f}")

print("\nClassification Report ATC:")
print(classification_report(true_ATC, pred_ATC, target_names=["ATC 0", "ATC 1"], zero_division=0))

print("\nClassification Report SAT:")
print(classification_report(true_SAT, pred_SAT, target_names=["SAT 0", "SAT 1"], zero_division=0))

print("\nClassification Report MAP:")
print(classification_report(true_MAP, pred_MAP, target_names=["MAP 0", "MAP 1"], zero_division=0))



print("Confusion Matrix: ATC\n", cm_ATC)
fig1 = plot_confusion_matrix(
    cm_ATC,
    name_task="ATC",
    name_classes=["class 0", "class 1"],
    save_path="confusion_ATC_MLT_LSTM.pdf"
)
fig2 = plot_confusion_matrix(
    cm_SAT,
    name_task="SAT",
    name_classes=["class 0", "class 1"],
    save_path="confusion_SAT_MLT_LSTM.pdf"
)
fig3 = plot_confusion_matrix(
    cm_MAP,
    name_task="MAP",
    name_classes=["class 0", "class 1"],
    save_path="confusion_MAP_MLT_LSTM.pdf"
)
plt.show()
plt.close(fig1)
plt.close(fig2)
plt.close(fig3)
